# 连续微磁模拟：一个 sim，分阶段输出

请在**Python (micromagnetics) 内核**运行此教程；不要重载、重启或中断正在计算的旧Notebook。主流程是3×3×1、仅各向异性、总动力学2fs的小示例，不包含退磁场、长时间计算或默认电流。它说明接口，不保证任意研究参数的科学精度。

类如`Grid`是构造类型；`grid = Grid(...)`产生实例。`model`、`initial`、`sim`是传递和持续使用的对象，字符串如`'drive'`只是输出名字。所有物理输入采用SI。先运行导入单元，再按顺序执行各单元；配置区的修改只在新建sim时生效。


In [ ]:
from pathlib import Path
from fimrun.micromagnetics import Grid, Sublattice, FiMMaterial, FiM, FiMAnisotropy, Uniform, Antiparallel, FiMSOT
from fimrun.notebook import NotebookSimulation


## 模型与初态

A/B是同一网格内两子晶格，不是上下两层。`m=M/Ms`是单位方向。`Antiparallel`显式生成B=−A，材料A/B仍可不同。只有放入`energy`的项参与计算。


In [ ]:
grid = Grid(cells=(3, 3, 1), cell_size=(4e-9, 4e-9, 3e-9))  # m
sub_A = Sublattice(Ms=3e5, alpha=0.01, gamma_H=2e5)
sub_B = Sublattice(Ms=6e5, alpha=0.01, gamma_H=2e5)
model = FiM(mesh=grid, material=FiMMaterial(A=sub_A, B=sub_B),
            energy=(FiMAnisotropy(K1_A=1e4, K1_B=2e4),))
initial = Antiparallel(Uniform((0., 0., 1.)))


## 创建一个模拟

`output_dir`是固定根目录；`output_name`只是本阶段的文件名，不是sim身份。显式重新执行构造会替换该根目录下的闲置sim并回到初态，旧输出仍保留；续算请继续调用同一sim。

下列目录固定，默认输出OVF。`magnetization=False`关闭磁态文件，`scalars=()`关闭CSV，两者均不关闭恢复state。每次run/relax均可设置output_name/保存间隔/scalars/magnetization，未填写则继承；动态save_interval与静态save_every_steps分别记忆，首次静态只存末态，显式save_interval=None也表示只存末态。


In [ ]:
output_dir = (Path.cwd() / 'results' / 'notebook_demo').resolve()
sim = NotebookSimulation(model=model, initial=initial, dt=1e-15,
    output_dir=output_dir, output_name='prepare', save_interval=1e-15,
    scalars=('time', 'mean_m'), magnetization=True, preview=False)
sim


## 动态预弛豫 → 静态 → 动态续算

动态`duration/dt/save_interval`用秒，必须对齐固定dt整步，不静默裁短步。动态阶段不会按静态门槛提前停止。静态save_every_steps是正整数接受迭代数，**不是秒**，不推进物理时间，也不增加动力学耗散/做功。

每个输出名称最多对应一个`<output_name>.csv`、一个`<output_name>_ovf/`和一个`<output_name>.state`，关闭的CSV/OVF不会生成。OVF含A/B单位方向、真实时间和阶段迭代信息。state可独立加载；没有多层segment目录和额外结果JSON。


In [ ]:
sim.run(1e-15, output_name='prepare', save_interval=1e-15)


### 静态两个条件AND

`residual_i=max|H_i−m_i(m_i·H_i)|` [A/m]；`mxh_i=max(|H_i−m_i(m_i·H_i)|/Ms_i)`无量纲，A/B各用自己的Ms，不能用净Ms。使用同一接受态的保守场。必须A/B两项都满足`residual`和`mxh`；连续满足consecutive个不同接受态才成功。`residual=1.0`是可覆盖默认值，mxh必须显式填写。

常量Ms时，各子晶格等效限制为`min(residual, Ms_i*mxh)`；默认1A/m有时更严格。例子的1e-5不是通用科学精度保证。新指标不同于旧BORIS更新前仅A指标；归档原稿阈值5e-5，手册例子有1e-5。


In [ ]:
sim.relax(mxh=1e-5, residual=1.0, consecutive=1,
          output_name='static', save_every_steps=1, scalars=('time', 'residual', 'mxh'))


In [ ]:
sim.run(1e-15, dt=1e-15, output_name='drive', save_interval=None,
        scalars=('time', 'mean_m', 'residual', 'mxh'))


## 同名覆盖与恢复

再次使用已有输出名时，先显示CSV/OVF/state的精确目标，输入`OVERWRITE`才覆盖；其它输入拒绝，且不计算、不改磁态/时间/dt/配置。没有默认追加。只替换经state校验归属的文件；外来文件或改过的结果拒绝覆盖，应选择新名字。取消科学运行会保存最后接受态；发布失败会报恢复位置并保留旧完整结果。跨文件不能承诺崩溃瞬间原子替换；隐藏恢复目录/锁用于失败恢复，遇到它们先检查，不能随意删除。

普通state重建求解历史，strict checkpoint才保留已支持算法的严格历史；两者不可互换。加载旧数据不迁移或删除原文件。以下手动操作可选，不在默认Run All执行：

```python
restored = NotebookSimulation.load(output_dir / 'drive.state',
    output_dir=(output_dir / 'restored').resolve(), output_name='continued')
restored.run(1e-15)
# 在接受态边界手动另存；同名拒绝，绝不覆盖。
sim.save('manual.state')
sim.save('manual.checkpoint', checkpoint=True)
# 如需释放资源，请明确执行；默认主流程留下sim供续算。
sim.close()
```


## 同页2D/3D预览（可选）

```python
sim.preview()
```

同一sim跨阶段复用标签页；替换实例时通过该服务的随机身份与递增绑定版本更新原页，旧帧仍拒绝。默认8766端口，冲突不会关闭别人的服务；需要别的端口时在构造中指定preview_port。3D标+X/+Y/+Z；2D为XY色图，X向右/Y向上，物理坐标nm和z层明确，可选mx/my/mz，色标[-1,1]。工具栏显示当前倍率、每个可见子晶格的实际箭头数和合计；2D显示采样格点数。抽稀只减少显示密度；箭头长度独立，不改求解网格。A+B表示同时展示两子晶格，不是向量求和。

网页显示A/B residual、mxh、各阈值与未过门、哪个限制更严。无同态缓存时显示不可用，不额外求场。保守场残差不是含SOT的完整动力学力矩；动态静态门槛未启用。视图切换/抽稀只使用缓存，归约、传输和绘图仍有成本。


## SOT可选示例：设置后再run

先定义一个电流实例，下面是完整六参数：

```python
sot = FiMSOT(J=(1e11, 0., 0.), xi_A=0.1, xi_B=-0.1,
             incident_direction=(0., 0., 1.), field_like_A=0., field_like_B=0.)
sim.set_current(sot)
sim.run(1e-15, output_name='sot')
```

J是传统电流密度[A/m²]，有限、均匀、恒定且Jz=0。xi_A/B为必填、有符号、有限无量纲效率；负xi翻转该子晶格SOT系数，不是负Ms或负电流大小。incident_direction默认(0,0,1)，非零方向会归一化，实际q=d×J，不能直接当作极化方向输入。field_like_A/B默认0，为有符号无量纲场混合系数；最终RHS有Gilbert混合，不能直接称最终DL/FL比。厚度来自cell_size[2]，A/B各用自己的Ms和gamma。当前不支持STT、AC、函数电流或GPU扩展。

要回到保守静态弛豫，先显式`sim.set_current(None)`，再`sim.relax(mxh=..., output_name='static_after_current')`。切换外场用`sim.set_field(B_ext=(0,0,-0.002))`，单位T；或`H_ext`单位A/m，二者不可同时传入。B_ext指外加μ0H，不是内部总B。


## 参数速查与旧接口

| 调用 | 参数、默认与限制 |
|---|---|
| NotebookSimulation | 必填model/initial/dt/output_dir；output_name='experiment'；save_interval=None仅末态；scalars=('time','mean_m')；magnetization=True默认OVF；preview=False；preview_port=8766；replace=True只允许替换闲置sim。根目录固定。 |
| run(duration, ...) | duration秒必填；dt=None继承；output_name/保存间隔/scalars/magnetization未填继承。save_interval动态秒，正有限且与dt对齐；None仅末态。dt变更后采样仍须对齐。 |
| relax(*, mxh, ...) | mxh无量纲必填；residual=1.0 A/m；consecutive=1正整数；阈值正有限；output_name/scalars/magnetization继承，save_every_steps静态正整数或None。仅现有FiM sdesc、无电流。旧位置tolerance不能当mxh，迁移错误发生在计算前。 |
| scalars | 元组，可选time/mean_m/energy/dissipation/drive_work/residual/mxh；空元组关闭CSV。m无量纲，能量/耗散/做功J，时间s，residual A/m，mxh无量纲。 |
| Grid | cells三正整数，cell_size三正有限米；或size按既有规则解析；origin默认(0,0,0)m，boundary默认open，mirrors默认(0,0)。限制与资格见PARAMETERS。 |
| Sublattice | Ms>0 A/m、alpha≥0无量纲、gamma_H>0 m/(A·s)，全部必填有限。FiMMaterial(A=...,B=...)必填；同网格双子晶格等半体积惯例。 |
| FiM/energy | mesh/material必填；energy=()默认无能量项。FiMExchange/OnsiteCoupling/DMI/FiMAnisotropy/Demag/Zeeman须显式加入，完整参数与限制见docs/interface-v01/PARAMETERS.md。 |
| 初态 | Uniform(direction)；Smooth(amplitude=...)；Skyrmion(center=...,radius=...,wall=...,helicity=0.,polarity=-1)米/弧度；Antiparallel包A；独立A/B字典；OVF成对文件需显式解释。完整签名见PARAMETERS。 |
| LegacyFixture('canonical_legacy') | 仅原125×125×1、4×4×3nm、零原点的解析初猜，不是已松弛态，材料另设。本3×3例子不能使用。 |
| 旧Case | workflow='prepare_then_drive'是预动力学→静态/资格门→fresh驱动；Initial(kind='canonical_legacy')选固定初猜。当前单sim流程显式顺序调用run/relax/set_current，不需要Case/workflow/solver/session。 |

旧results/name构造作为显式兼容路径保留原目录布局和A/m tolerance语义；新教程统一使用output_dir/output_name，不能混用两套参数。更改源码后必须另选新内核，本教程不会自动reload或重启旧内核。


## 模型参数完整附录

下面列出每个模型、能量项和初态构造器的参数、单位、默认和限制。按需从fimrun.micromagnetics导入相应类；不在主流程默认启用这些项。运行与输出使用上方Notebook新接口，勿混用核心Outputs的默认语义。

## Geometry, model and material

| Constructor | Fields / units / defaults | Conditions and mutation |
|---|---|---|
| Grid | exactly one of size: 3 positive sample lengths [m] or cells: 3 integer counts; cell_size: 3 positive lengths [m] required; origin=(0,0,0) [m]; boundary='open'; mirrors=(0,0) | Single nz=1. Implemented n 1..256, aspect <=4, mirrors 0..40; these are geometry/implementation support domains, not physics laws. Open requires zero mirrors; periodic_xy requires nx,ny>=3. No region masks/holes in this release: every cell is magnetic. New experiment for grid/boundary/model conversion. |
| FM | mesh:Grid, material:FMMaterial required; energy=() | Exactly one direction field and full-volume energy. Open only. |
| FiM | mesh:Grid, material:FiMMaterial required; energy=() | Two independent fields, open or periodic_xy. |
| FMMaterial / Sublattice | Ms [A/m], alpha [dimensionless], gamma_H [m/(A s)], all required | Ms/gamma>0, alpha>=0, finite; gamma_H is Gilbert H convention, not rad/(T s). Constants only. |
| FiMMaterial | A:Sublattice, B:Sublattice required; convention='equal_half_volume' | No implicit B. No alternative volume fractions. Material changes require branch(model=...). |

`M=Ms*m` for FM. FiM uses `Mbar=(Ms_A*m_A+Ms_B*m_B)/2`; local self energies and dissipation/work use half-volume weights. See CONTRACT.md and the unchanged MODEL_AND_UNITS.md convention. m is dimensionless, physical cell volume is the product of cell_size. No default material or legacy field is supplied.

## Explicit energy and torque terms

Omitted terms do not participate. Duplicate/inapplicable terms are rejected. Use a new model on a branch to change terms/material; `set_field` is the supported same-experiment field switch. The local kernel computes field and fused energy together; disabling energy curve output suppresses recording, not the field's shared bookkeeping needed for physical state/energy-switch accounting.

| Constructor | Parameters | Models / equations / dependencies |
|---|---|---|
| Exchange | A>=0 [J/m], required | FM: bond energy V*A*|delta m|²/h²; H bond=2A*delta m/(mu0*Ms*h²). |
| FiMExchange | A_A,A_B>=0 [J/m] required; Anh=0 [J/m] | FiM: each self bond V/2*A_i*|delta m_i|²/h²; cross .5*V*Anh*dot(delta m_A,delta m_B)/h². H self factor2A_i and cross Anh over own mu0*Ms_i*h². Nonzero Anh requires periodic_xy and abs(Anh)<2*sqrt(A_A*A_B). This term groups A and Anh execution. |
| OnsiteCoupling | Ah [J/m³] required, signed | FiM: E=-2*Ah*V*dot(m_A,m_B); H_i=4*Ah*m_other/(mu0*Ms_i). Negative favors antiparallel with this convention. |
| DMI | D_A,D_B [J/m²] required, signed | FiM only. Positive-neighbor XY bond E_i=.5*V*D_i*(m_i[d]*neighbor.z-m_i.z*neighbor[d])/h_d, fields are the existing stencil derivative with own Ms. No FM DMI. |
| UniaxialAnisotropy | K1 [J/m³] required; axis=(0,0,1) | FM: E=V*K1*(1-u²), H=2*K1*u*axis/(mu0*Ms). Nonzero axis normalized explicitly in configuration. No FM K2. |
| FiMAnisotropy | K1_A,K1_B required; K2_A=K2_B=0 [J/m³]; axis_A=axis_B=(0,0,1) | FiM: E_i=V/2*(K1_i*s+K2_i*s²), s=1-u_i²; H_i=2*u_i*(K1_i+2*K2_i*s)*axis_i/(mu0*Ms_i). |
| Demag | no arguments | Existing finite-volume Newell/asymptotic tensor, FFT CPU; FM uses its existing Accelerate single-field convolution, FiM uses existing serial FFTW shared Mbar. Open/finite-image periodic support as above. No allocation/generation/FFT when omitted. Kernels retain original reference gates (tensor error<=1e-10, drift/reciprocity<=1e-12); engineering checks are not continuum or infinite-PBC certificates. |
| Zeeman | exactly one H_ext [A/m] or B_ext [T], required | FM/FiM: external B_ext=mu0*H_ext. Omitted means zero field. set_field(...) inserts/replaces term, records same-m energy jump. No arbitrary time function. |
| FiMSOT | J=(Jx,Jy,0) [A/m²], xi_A,xi_B required; incident_direction=(0,0,1), field_like_A=field_like_B=0 | FiM only, uniform conventional current, q=incident_direction cross J. Existing effective field=-C/gamma_H*(m cross q+r*q), C=xi*(muB/e)/(Ms*cell_thickness); existing Gilbert RHS and work. No electrical transport/STT/AC/pulse. None disables; zero J/xi or q skips SOT contribution/work. `with_current_density(J=...)` returns a validated replacement. |

## Initial magnetization

Initial state is required; not providing it is an error. All arrays have shape `(nz,ny,nx,3)` per field, x fastest, finite nonzero unit directions. `sim.m` returns a copy of shape `(1,nz,ny,nx,3)` FM or `(2,nz,ny,nx,3)` FiM. FiM accepts exactly `{'A': ..., 'B': ...}` or explicit Antiparallel. No net-magnetization reconstruction.

| Constructor | Fields | Meaning |
|---|---|---|
| Uniform | direction required; normalize=False | Explicit normalization opt-in; otherwise unit error<=1e-12 required. |
| Smooth | amplitude required, abs<1 | Existing cosine XY profile about +z. |
| Skyrmion | center=(x,y) [m], radius>0 [m], wall>0 [m] required; helicity=0 [rad], polarity=-1 | Existing tanh radial single-texture formula; polarity +/-1. Only these fields are accepted. |
| Antiparallel | A constructor/array required | Explicitly build B=-A, only FiM. |
| LegacyFixture | name required | Only 'canonical_legacy', canonical125/4x4x3nm/origin0; material is still explicit. Historical fixture, not generic equilibrium or material qualification. |
| OVF | path, quantity required | quantity='direction' or 'magnetization_Apm'; existing strict rectangular single-segment OVF2 Text/Binary4/Binary8. Grid/base/spacing must match, no interpolation. Ms units/magnitudes validated for magnetization_Apm. Loading is fresh m only, not state/time/material. |

The equivalent factory names are `Magnetization.uniform/smooth/skyrmion/antiparallel/legacy/ovf`.

